# Stage 2 — Multi-Scale Directional Feature Extraction

Runs three complementary filters on Stage 1's output: **Frangi Hessian vesselness**, a **matched filter bank** (Gaussian & Cauchy), and a **Gabor wavelet bank**. Each produces its own response map — fusing them into one composite map is deliberately left for a later step, so each can be inspected on its own first.

**Before running**: point `IMAGE_PATH`/`MASK_PATH` at a real DRIVE (or STARE/CHASE_DB1) image, same as notebook `01_stage1_preprocessing.ipynb`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.io_utils import load_fundus_image, load_fov_mask
from src.preprocessing import run_stage1
from src.feature_extraction import (
    frangi_vesselness, matched_filter_bank, gabor_filter_bank, run_stage2,
)

In [ ]:
IMAGE_PATH = str(PROJECT_ROOT / "data" / "drive" / "images" / "21_training.tif")
MASK_PATH = str(PROJECT_ROOT / "data" / "drive" / "mask" / "21_training_mask.gif")
# STARE / CHASE_DB1 don't ship a mask -- pass MASK_PATH = None for those.

rgb = load_fundus_image(IMAGE_PATH)
fov_mask = load_fov_mask(MASK_PATH)
stage1_out = run_stage1(rgb, fov_mask)
print("Stage 1 output:", stage1_out.shape, stage1_out.dtype, stage1_out.min(), stage1_out.max())

In [ ]:
# --- Tunable stage-2 parameters ---
SIGMAS = range(1, 5, 1)      # Frangi scale range
MF_SIGMA = 2.0                # matched-filter cross-section width
MF_LENGTH = 9                 # matched-filter length along the vessel axis
N_ORIENTATIONS = 12           # shared by matched-filter and Gabor banks (15-degree steps)
GABOR_FREQUENCY = 0.15

maps = run_stage2(
    stage1_out,
    frangi_kwargs={"sigmas": SIGMAS},
    matched_kwargs={"sigma": MF_SIGMA, "length": MF_LENGTH, "n_orientations": N_ORIENTATIONS},
    gabor_kwargs={"frequency": GABOR_FREQUENCY, "n_orientations": N_ORIENTATIONS},
)
for name, m in maps.items():
    print(f"{name:16s} shape={m.shape} dtype={m.dtype} min={m.min():.4f} max={m.max():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 5))
panels = [
    (stage1_out, "Stage 1 output"),
    (maps["frangi"], "Frangi vesselness"),
    (maps["matched_gaussian"], "Matched filter (Gaussian)"),
    (maps["matched_cauchy"], "Matched filter (Cauchy)"),
    (maps["gabor"], "Gabor wavelet bank"),
]
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Visual checks for each map (no fused output yet)
- **Frangi**: thin capillaries near the FOV edge should light up continuously; the optic-disc margin should stay dark (if it lights up, narrow `SIGMAS`).
- **Matched filter (both kernels)**: vessel centerlines should be continuous bright ridges at *every* orientation present in the image, not just horizontal/vertical; flat background/optic disc should be near-zero (confirms mean-subtraction is working correctly).
- **Gabor**: similar continuity check, but expect it to pick up the thinnest capillaries best — if it's mostly picking up noise instead, lower `GABOR_FREQUENCY` or increase `N_ORIENTATIONS`.

In [ ]:
# Optional: live parameter tuning with sliders (requires ipywidgets)
from ipywidgets import interact, FloatSlider, IntSlider

@interact(
    mf_sigma=FloatSlider(value=MF_SIGMA, min=0.5, max=6.0, step=0.5),
    frequency=FloatSlider(value=GABOR_FREQUENCY, min=0.05, max=0.4, step=0.01),
    n_orientations=IntSlider(value=N_ORIENTATIONS, min=4, max=24, step=2),
)
def tune(mf_sigma=MF_SIGMA, frequency=GABOR_FREQUENCY, n_orientations=N_ORIENTATIONS):
    mf = matched_filter_bank(stage1_out, sigma=mf_sigma, length=MF_LENGTH, n_orientations=n_orientations)
    gb = gabor_filter_bank(stage1_out, frequency=frequency, n_orientations=n_orientations)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(mf, cmap="gray"); axes[0].set_title("Matched filter"); axes[0].axis("off")
    axes[1].imshow(gb, cmap="gray"); axes[1].set_title("Gabor"); axes[1].axis("off")
    plt.show()

In [ ]:
# Save the 5-panel comparison figure for later reference / reports
out_dir = PROJECT_ROOT / "outputs" / "stage2"
out_dir.mkdir(parents=True, exist_ok=True)

image_id = Path(IMAGE_PATH).stem
fig.savefig(out_dir / f"{image_id}_stage2.png", dpi=150, bbox_inches="tight")
print("Saved to", out_dir / f"{image_id}_stage2.png")